# DATA PROCESSING: CESM2-LENS Fixed-Depth Heat Budget

## Import packages and define functions

In [1]:
import cmocean
import numpy as np
import xarray as xr
import pandas as pd
import cftime
import dask
import matplotlib.pyplot as plt
import os
import xesmf as xe
import pop_tools

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter,LatitudeFormatter
from cartopy.util import add_cyclic_point
import matplotlib
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import matplotlib.dates as mdates

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.animation import FFMpegWriter as fw
from matplotlib.animation import PillowWriter as pw
import json 
from IPython.utils import io
from os.path import exists
from scipy.stats import gaussian_kde
import seaborn as sns
import netCDF4 as nc4
from tqdm import tqdm
import pickle
import bottleneck

/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/pop_tools/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
from tqdm import tqdm

In [3]:
# Accessing glade CESM LENS2
def get_var_paths(directory, var):
    # Prefixes to match for future and historical datasets
    prefixes_to_match_fut = ['b.e21.BSSP370cmip6.', 'b.e21.BSSP370smbb.']
    prefixes_to_match_hist = ['b.e21.BHISTcmip6.', 'b.e21.BHISTsmbb.']

    # Sets to store unique prefixes for future and historical filenames
    prefixes_fut = list()
    prefixes_hist = list()

    # Iterate through files in the directory
    for filename in os.listdir(directory):
        # Check and add prefixes for future scenario files
        if any(filename.startswith(prefix) for prefix in prefixes_to_match_fut) and filename.endswith('.nc'):
            prefixes_fut.append(filename.rsplit('.', 3)[0])

        # Check and add prefixes for historical scenario files
        if any(filename.startswith(prefix) for prefix in prefixes_to_match_hist) and filename.endswith('.nc'):
            prefixes_hist.append(filename.rsplit('.', 3)[0])

    prefixes_hist_set = set(prefixes_hist)
    sorted_unique_list_hist = sorted(prefixes_hist_set)

    prefixes_fut_set = set(prefixes_fut)
    sorted_unique_list_fut = sorted(prefixes_fut_set)

    path_intermed_fut = sorted_unique_list_fut
    path_intermed_hist = sorted_unique_list_hist

    return path_intermed_hist, path_intermed_fut

def get_ds_var(directory, var, comp, index_hist):
    path_intermed_hist, path_intermed_fut = get_var_paths(directory, var)
    filename_identifier = '.'.join(path_intermed_hist[index_hist].rsplit('.', 5)[1:4])
    index_fut = find_identifier_with_index(path_intermed_hist, filename_identifier)[0][1]
    hist_file_paths = get_hist_file_paths(var,directory, path_intermed_hist, index_hist)
    fut_file_paths = get_fut_file_paths(var, directory, path_intermed_fut, index_fut)
    ds_var_fut = file_path_to_var_ds(fut_file_paths)
    ds_var_hist = file_path_to_var_ds(hist_file_paths)
    return ds_var_hist, ds_var_fut

def find_identifier_with_index(prefixes, identifier):
    """
    Find prefixes that contain a specific identifier and their indices.

    Parameters:
        prefixes (list): A list of prefixes to search through.
        identifier (str): The identifier to search for in the prefixes.

    Returns:
        list: A list of tuples containing matching prefixes and their indices.
    """
    matching_prefixes_with_indices = []  # Initialize a list to store matches and their indices

    for index, prefix in enumerate(prefixes):  # Use enumerate to get both index and prefix
        if identifier in prefix:  # Check if the identifier is in the prefix
            matching_prefixes_with_indices.append((prefix, index))  # Add the prefix and index as a tuple

    return matching_prefixes_with_indices  # Return the list of matching prefixes and indices

def get_hist_file_paths(var, directory, path_intermed_hist, index):
    attrib_title = path_intermed_hist[index]
    file_paths = []
    for start_year in range(1850, 2010, 10):
        end_year = start_year + 9
        file_path = f'{directory}{attrib_title}.{var}.{start_year}01-{end_year}12.nc'
        file_paths.append(file_path)
    last_file_path = f'{directory}{attrib_title}.{var}.201001-201412.nc'
    file_paths.append(last_file_path)
    return file_paths

def get_fut_file_paths(var, directory, path_intermed_fut, index):
    attrib_title = path_intermed_fut[index]
    file_paths = []
    for start_year in range(2015, 2095, 10):
        end_year = start_year + 9
        file_path = f'{directory}{attrib_title}.{var}.{start_year}01-{end_year}12.nc'
        file_paths.append(file_path)
    last_file_path = f'{directory}{attrib_title}.{var}.209501-210012.nc'
    file_paths.append(last_file_path)
    return file_paths

def file_path_to_var_ds(file_paths):
    var_ds = xr.open_mfdataset(file_paths,
                                 concat_dim='time',
                                 combine='nested',
                                 parallel=True)
    return var_ds

def process_cesm_lens2(var, comp='ocn', ens_memb_index=0, 
                      hist_time_slice=('1979-01-01', '2015-01-01'),
                      fut_time_slice=('2015-02-01', '2020-12-01'),
                      z_slice=slice(0, 20),
                      lat_slice=slice(204, 367),
                      lon_slice=slice(190, 280)):
    """
    Process CESM LENS2 data for a given variable.
    
    Parameters:
    -----------
    var : str
        Variable name (e.g., 'TEND_TEMP')
    comp : str, optional
        Component (default: 'ocn')
    ens_memb_index : int, optional
        Ensemble member index (default: 0)
    hist_time_slice : tuple, optional
        Historical period time slice (default: ('1979-01-01', '2015-01-01'))
    fut_time_slice : tuple, optional
        Future period time slice (default: ('2015-02-01', '2020-12-01'))
    z_slice : slice, optional
        Depth slice (default: slice(0, 20))
    lat_slice : slice, optional
        Latitude slice (default: slice(204, 367))
    lon_slice : slice, optional
        Longitude slice (default: slice(190, 280))
    
    Returns:
    --------
    xarray.DataArray
        Processed data array
    """
    
    # Construct directory path
    directory = f'/glade/campaign/cgd/cesm/CESM2-LE/{comp}/proc/tseries/month_1/{var}/'
    
    # Get historical and future datasets
    ds_var_hist, ds_var_fut = get_ds_var(directory, var, comp, ens_memb_index)
    
    # Slice the time periods and depth
    hist_sliced = ds_var_hist[var].sel(time=slice(*hist_time_slice)).isel(z_t=z_slice)
    fut_sliced = ds_var_fut[var].sel(time=slice(*fut_time_slice)).isel(z_t=z_slice)
    
    # Concatenate along time dimension
    combined_ds = xr.concat([hist_sliced, fut_sliced], dim='time')
    
    # Select spatial domain and compute
    processed_data = combined_ds[:, :, lat_slice, lon_slice].compute()
    
    return processed_data

## Fixed-depth heat budget

In [7]:
filepath_g = f"/glade/derecho/scratch/cassiacai/daily_surface_potential_temperature.nc" #pop_tools.DATASETS.fetch('daily_surface_potential_temperature.nc')
ds_g = xr.open_dataset(filepath_g)

/glade/derecho/scratch/cmendiola/tmp/ipykernel_79893/2263814207.py:2: FutureWarning: In a future version, xarray will not decode the variable 'days_in_norm_year' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds_g = xr.open_dataset(filepath_g)


In [8]:
NO_SEC_IN_1_MON = 2629740

hist_time_slice=('1979-01-01', '2015-01-01')
fut_time_slice=('2015-02-01', '2020-12-01')
z_slice=slice(0, 20),
# lat_slice=slice(204, 367)
# lon_slice=slice(190, 280)
comp = 'ocn'

In [9]:
%%time
for ens_memb_index in range(83, 100):
    print(ens_memb_index)
    heat_budget_terms = xr.Dataset()
    # # variables
    # var = 'TEND_TEMP'
    # directory = f'/glade/campaign/cgd/cesm/CESM2-LE/{comp}/proc/tseries/month_1/{var}/'
    # ds_var_hist, ds_var_fut = get_ds_var(directory, var, comp, ens_memb_index)
    # hist_sliced = ds_var_hist[var].sel(time=slice('1979-01-01', '2015-01-01')).isel(z_t=slice(0, 15))
    # fut_sliced = ds_var_fut[var].sel(time=slice('2015-02-01', '2020-12-01')).isel(z_t=slice(0, 15))
    # combined_ds = xr.concat([hist_sliced, fut_sliced], dim='time')
    # heat_budget_terms[var] = combined_ds[:, :, :, :].compute()
    # # heat_budget_terms[var] = combined_ds[:, :, lat_slice, lon_slice].compute()
    # print(var)
    
    var = 'UET'
    directory = f'/glade/campaign/cgd/cesm/CESM2-LE/{comp}/proc/tseries/month_1/{var}/'
    ds_var_hist, ds_var_fut = get_ds_var(directory, var, comp, ens_memb_index)
    hist_sliced = ds_var_hist[var].sel(time=slice('1979-01-01', '2015-01-01')).isel(z_t=slice(0, 15))
    fut_sliced = ds_var_fut[var].sel(time=slice('2015-02-01', '2020-12-01')).isel(z_t=slice(0, 15))
    combined_ds = xr.concat([hist_sliced, fut_sliced], dim='time')
    heat_budget_terms[var] = combined_ds[:, :, :, :].compute()
    # heat_budget_terms[var] = combined_ds[:, :, lat_slice, lon_slice].compute()
    print(var)
    
    var = 'VNT'
    directory = f'/glade/campaign/cgd/cesm/CESM2-LE/{comp}/proc/tseries/month_1/{var}/'
    ds_var_hist, ds_var_fut = get_ds_var(directory, var, comp, ens_memb_index)
    hist_sliced = ds_var_hist[var].sel(time=slice('1979-01-01', '2015-01-01')).isel(z_t=slice(0, 15))
    fut_sliced = ds_var_fut[var].sel(time=slice('2015-02-01', '2020-12-01')).isel(z_t=slice(0, 15))
    combined_ds = xr.concat([hist_sliced, fut_sliced], dim='time')
    heat_budget_terms[var] = combined_ds[:, :, :, :].compute()
    # heat_budget_terms[var] = combined_ds[:, :, lat_slice, lon_slice].compute()
    print(var)
    
    var = 'WTT'
    directory = f'/glade/campaign/cgd/cesm/CESM2-LE/{comp}/proc/tseries/month_1/{var}/'
    ds_var_hist, ds_var_fut = get_ds_var(directory, var, comp, ens_memb_index)
    hist_sliced = ds_var_hist[var].sel(time=slice('1979-01-01', '2015-01-01')).isel(z_w_top=slice(0, 15))
    fut_sliced = ds_var_fut[var].sel(time=slice('2015-02-01', '2020-12-01')).isel(z_w_top=slice(0, 15))
    combined_ds = xr.concat([hist_sliced, fut_sliced], dim='time')
    heat_budget_terms[var] = combined_ds[:, :, :, :].compute()
    # heat_budget_terms[var] = combined_ds[:, :, lat_slice, lon_slice].compute()
    print(var)
    
    var = 'QSW_3D'
    directory = f'/glade/campaign/cgd/cesm/CESM2-LE/{comp}/proc/tseries/month_1/{var}/'
    ds_var_hist, ds_var_fut = get_ds_var(directory, var, comp, ens_memb_index)
    hist_sliced = ds_var_hist[var].sel(time=slice('1979-01-01', '2015-01-01')).isel(z_w_top=slice(0, 15))
    fut_sliced = ds_var_fut[var].sel(time=slice('2015-02-01', '2020-12-01')).isel(z_w_top=slice(0, 15))
    combined_ds = xr.concat([hist_sliced, fut_sliced], dim='time')
    heat_budget_terms[var] = combined_ds[:, :, :, :].compute()
    # heat_budget_terms[var] = combined_ds[:, :, lat_slice, lon_slice].compute()
    print(var)
    
    var = 'SHF'
    directory = f'/glade/campaign/cgd/cesm/CESM2-LE/{comp}/proc/tseries/month_1/{var}/'
    ds_var_hist, ds_var_fut = get_ds_var(directory, var, comp, ens_memb_index)
    hist_sliced = ds_var_hist[var].sel(time=slice('1979-01-01', '2015-01-01'))
    fut_sliced = ds_var_fut[var].sel(time=slice('2015-02-01', '2020-12-01'))
    combined_ds = xr.concat([hist_sliced, fut_sliced], dim='time')
    heat_budget_terms[var] = combined_ds[:, :, :].compute()
    # heat_budget_terms[var] = combined_ds[:, lat_slice, lon_slice].compute()
    print(var)

    var = 'SHF_QSW'
    directory = f'/glade/campaign/cgd/cesm/CESM2-LE/{comp}/proc/tseries/month_1/{var}/'
    ds_var_hist, ds_var_fut = get_ds_var(directory, var, comp, ens_memb_index)
    hist_sliced = ds_var_hist[var].sel(time=slice('1979-01-01', '2015-01-01'))
    fut_sliced = ds_var_fut[var].sel(time=slice('2015-02-01', '2020-12-01'))
    combined_ds = xr.concat([hist_sliced, fut_sliced], dim='time')
    heat_budget_terms[var] = combined_ds[:, :, :].compute()
    # heat_budget_terms[var] = combined_ds[:, lat_slice, lon_slice].compute()
    print(var)
    
    var = 'DIA_IMPVF_TEMP'
    directory = f'/glade/campaign/cgd/cesm/CESM2-LE/{comp}/proc/tseries/month_1/{var}/'
    ds_var_hist, ds_var_fut = get_ds_var(directory, var, comp, ens_memb_index)
    hist_sliced = ds_var_hist[var].sel(time=slice('1979-01-01', '2015-01-01')).isel(z_w_bot=slice(0, 15))
    fut_sliced = ds_var_fut[var].sel(time=slice('2015-02-01', '2020-12-01')).isel(z_w_bot=slice(0, 15))
    combined_ds = xr.concat([hist_sliced, fut_sliced], dim='time')
    heat_budget_terms[var] = combined_ds[:, :, :, :].compute()
    # heat_budget_terms[var] = combined_ds[:, :, lat_slice, lon_slice].compute()
    print(var)

    heat_budget_terms['hflux_factor'] = ds_var_hist['hflux_factor'][0].item()

    # DZT = np.zeros((20,163,90))
    DZT = np.zeros((15,384,320))

    for k in range(15):
        DZT[k,:,:] = heat_budget_terms.z_t[k]
    
    ds2 = xr.Dataset({'DZT': (['z_t','nlat','nlon',], DZT),
                       'DZU': (['z_t','nlat','nlon'], DZT)},
                        coords={'TLAT': (['nlat','nlon'],heat_budget_terms.TLAT.data),
                                'TLONG': (['nlat','nlon'],heat_budget_terms.TLONG.data),
                                'ULAT': (['nlat','nlon'],heat_budget_terms.ULAT.data),
                                'ULONG': (['nlat','nlon'],heat_budget_terms.ULONG.data),
                                'z_t': (['z_t'],heat_budget_terms.z_t.data)})

    heat_budget_terms = heat_budget_terms.assign_coords(z_w=ds_g.z_w[:15])
    heat_budget_terms = heat_budget_terms.assign_coords(z_w_bot=ds_g.z_w_bot[:15])
    heat_budget_terms["DXU"] = ds_g.DXU#[lat_slice, lon_slice]
    heat_budget_terms["DYU"] = ds_g.DYU#[lat_slice, lon_slice]
    heat_budget_terms["DXT"] = ds_g.DXT#[lat_slice, lon_slice]
    heat_budget_terms["DYT"] = ds_g.DYT#[lat_slice, lon_slice]
    heat_budget_terms["UAREA"] = ds_g.UAREA#[lat_slice, lon_slice]
    heat_budget_terms["TAREA"] = ds_g.TAREA#[lat_slice, lon_slice]
    heat_budget_terms["DZT"] = ds2.DZT
    heat_budget_terms["DZU"] = ds2.DZU
    heat_budget_terms["dz"] = heat_budget_terms.z_t
    
    heat_budget_terms.DZT.attrs["long_name"] = "Thickness of T cells"
    heat_budget_terms.DZT.attrs["units"] = "centimeter"
    heat_budget_terms.DZT.attrs["grid_loc"] = "3111"
    heat_budget_terms.DZU.attrs["long_name"] = "Thickness of U cells"
    heat_budget_terms.DZU.attrs["units"] = "centimeter"
    heat_budget_terms.DZU.attrs["grid_loc"] = "3221"

    VOL = (heat_budget_terms.DZT * heat_budget_terms.DXT * heat_budget_terms.DYT).compute()
    # KMT = ds_g.KMT[lat_slice, lon_slice].compute()
    KMT = ds_g.KMT[:, :].compute()
    
    for j in tqdm(range(len(KMT.nlat))):
        for i in range(len(KMT.nlon)):
            k = KMT.values[j, i].astype(int)
            VOL.values[k:, j, i] = 0.0
    
    heat_budget_terms["VOL"] = VOL
    
    heat_budget_terms.VOL.attrs["long_name"] = "volume of T cells"
    heat_budget_terms.VOL.attrs["units"] = "centimeter^3"
    heat_budget_terms.VOL.attrs["grid_loc"] = "3111"

    budget = xr.Dataset()
    metrics = {
        ("X",): ["DXU", "DXT"],  # X distances
        ("Y",): ["DYU", "DYT"],  # Y distances
        ("Z",): ["DZU", "DZT"],  # Z distances
        ("X", "Y"): ["UAREA", "TAREA"],
    }
    
    gridxgcm, dsxgcm = pop_tools.to_xgcm_grid_dataset(
        heat_budget_terms,
        periodic=False,
        metrics=metrics,
        boundary={"X": "extend", "Y": "extend", "Z": "extend"},
    )
      
    for coord in ["nlat", "nlon"]:
        if coord in dsxgcm.coords:
            dsxgcm = dsxgcm.drop_vars(coord)
    
    # budget['TEND_TEMP'] = dsxgcm.TEND_TEMP
    budget_intermed = xr.Dataset()
    budget_intermed["UET"] = -(gridxgcm.diff(dsxgcm.UET * dsxgcm.VOL.values, axis="X") / dsxgcm.VOL)
    budget_intermed["VNT"] = -(gridxgcm.diff(dsxgcm.VNT * dsxgcm.VOL.values, axis="Y") / dsxgcm.VOL)
    budget_intermed["WTT"] = (
        gridxgcm.diff(dsxgcm.WTT.fillna(0) * (dsxgcm.dz * dsxgcm.DXT * dsxgcm.DYT).values, axis="Z")
        / dsxgcm.VOL
    )
    budget["TOT_ADV"] = budget_intermed["UET"] + budget_intermed["VNT"] + budget_intermed["WTT"]
    budget["DIA_IMPVF_TEMP"] = -(gridxgcm.diff(dsxgcm.DIA_IMPVF_TEMP * dsxgcm.TAREA, axis="Z") / dsxgcm.VOL)
    SRF_TEMP_FLUX = (dsxgcm.SHF) * dsxgcm.hflux_factor # do not remove SRW
    SRF_TEMP_FLUX = SRF_TEMP_FLUX.load()
    first_term = (SRF_TEMP_FLUX*dsxgcm.TAREA)/dsxgcm.VOL.values[0, :, :]
    budget["SHF_depth"] = budget["DIA_IMPVF_TEMP"].copy()
    budget["SHF_depth"][:] = 0
    budget["SHF_depth"][:,0,:,:] = first_term
    budget["QSW_3D"] = -gridxgcm.diff((dsxgcm.QSW_3D * dsxgcm.hflux_factor), axis="Z") / dsxgcm.DZT
    # budget['SUBGRID'] = budget['TEND_TEMP'] - budget['SHF_depth'] - budget["DIA_IMPVF_TEMP"] - budget['TOT_ADV'] - budget['QSW_3D']
    filepath = f"/glade/derecho/scratch/cmendiola/heat_budget_blob_fosi_lens2_paper/heat_budget_{ens_memb_index}_03172026.nc"
    budget.sum(dim='z_t').to_netcdf(filepath)
    print(filepath)
    print('***************************')

83


/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/dask/_task_spec.py:758: FutureWarning: In a future version, xarray will not decode the variable 'days_in_norm_year' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  return self.func(*new_argspec, **kwargs)
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/dask/_task_spec.py:758: FutureWarning: In a future version, xarray will not decode the variable 'days_in_norm_year' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instea

UET


/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/dask/_task_spec.py:758: FutureWarning: In a future version, xarray will not decode the variable 'days_in_norm_year' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  return self.func(*new_argspec, **kwargs)
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/dask/_task_spec.py:758: FutureWarning: In a future version, xarray will not decode the variable 'days_in_norm_year' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instea

VNT


...

SHF_QSW


/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/dask/_task_spec.py:758: FutureWarning: In a future version, xarray will not decode the variable 'days_in_norm_year' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  return self.func(*new_argspec, **kwargs)
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/dask/_task_spec.py:758: FutureWarning: In a future version, xarray will not decode the variable 'days_in_norm_year' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instea

DIA_IMPVF_TEMP


100%|██████████| 384/384 [00:00<00:00, 1531.27it/s]


/glade/derecho/scratch/cmendiola/heat_budget_blob_fosi_lens2_paper/heat_budget_99_03172026.nc
***************************
CPU times: user 51min 7s, sys: 7min 52s, total: 58min 59s
Wall time: 2h 34min 31s


In [ ]:
# filepath = f"/glade/derecho/scratch/cassiacai/heat_budget_blob_fosi_lens2_paper/heat_budget_{ens_memb_index}_05232025.nc"
# budget.sum(dim='z_t').to_netcdf(filepath)
# print(filepath)

In [ ]:
u/home/cassiacai/measures/heatbudget_computeLENS2heatbudget.ipynb